File `extraction_azure.ipynb` changelog
- 2025 June 27 - v0.0.1 define services & plan of attack, begin env setup - KGK

# MicroSoft Azure-based data extraction
EasyOCR was found to have poor performance transcribing the handwritten forms. EasyOCR is in widespread support with many glowing reviews, so it may be that we can fine-tune our utilization for better results. However, we are also exploring Google Vision (see the header "Alternative (Premium) OCR" in `extraction.ipynb`) and, here, Microsoft Azure.

## Creating an Azure Account and Connecting to Services
To create an Azure account, visit [this link](https://azure.microsoft.com/en-us/pricing/purchase-options/azure-account). Free plans are available which grant twelve months of access to the "AI Vision" service we'll be using. Inside Azure's interface, it's called "computer vision".

Once you've created an account and are in the Azure dashboard, search "computer vision" and hit the "create computer vision" button. I selected my "Azure subscription 1" and created a new resource group named "azure_stormwater_exp". Under the Free tier "F0", I'm allowed 20 calls per minute/5k calls per month. I made a new instance called "stormwater-form-reader" with the F0 tier plan. This required agreeing to the [Microsoft Product and Services Data Protection Addendum (WW) (*download link*)](https://www.microsoft.com/licensing/docs/documents/download/MicrosoftProductandServicesDPA(WW)(English)(April2025)(CR).docx) and [Transparency note for Spatial Analysis](https://learn.microsoft.com/en-us/legal/cognitive-services/computer-vision/transparency-note-spatial-analysis?context=%2Fazure%2Fcognitive-services%2Fcomputer-vision%2Fcontext%2Fcontext#considerations-when-choosing-a-use-case).

### Network Settings
Selected "All networks, including the internet, can access this resource."

### Managed Identity
Neither a system-assigned or user-assigned managed identity was established for this resource. This will limit it's ability to interact with other services, e.g. storage accounts. For our MVP, this is not required.

### Tags
No tags were assigned.

## The ChatGPT Game Plan
Kaleb provided the following general outline from ChatGPT which was generated in the context of his ongoing research on the topic:

> You’re welcome! Here’s a sample Python script to use the Azure AI Vision Read API v3.2 to extract OCR text with confidence scores, including handwriting, from an image.

### 🛠️ Prerequisites
1. Azure Subscription: Set up free tier here.
2. Create a Computer Vision resource in Azure Portal.
3. Note your:  
    - Endpoint URL (e.g., https://<your-region>.api.cognitive.microsoft.com/)
    - Subscription Key
4. Install the Azure SDK for Vision:
    - pip install azure-cognitiveservices-vision-computervision

---

### 📜 Python Script: Azure OCR with Confidence Scores

``` python
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from msrest.authentication import CognitiveServicesCredentials
import time
import os
```

### 🔐 Set your credentials
``` python
subscription_key = "YOUR_SUBSCRIPTION_KEY"
endpoint = "YOUR_ENDPOINT_URL"  # e.g. "https://eastus.api.cognitive.microsoft.com/"
```

### 🔧 Initialize client
``` python
client = ComputerVisionClient(endpoint, CognitiveServicesCredentials(subscription_key))
```

### 📷 Image path (local)
``` python
image_path = "path/to/your-image.jpg"
```

### 🔁 Read API call (asynchronous)
``` python
with open(image_path, "rb") as image_stream:
    read_response = client.read_in_stream(image_stream, raw=True)
```

### ⏳ Get operation ID
``` python
operation_location = read_response.headers["Operation-Location"]
operation_id = operation_location.split("/")[-1]
```

### ⏱️ Poll for result
``` python
while True:
    read_result = client.get_read_result(operation_id)
    if read_result.status not in [OperationStatusCodes.not_started, OperationStatusCodes.running]:
        break
    time.sleep(1)
```

### ✅ Parse results
``` python
if read_result.status == OperationStatusCodes.succeeded:
    for page in read_result.analyze_result.read_results:
        print(f"\n📄 Page {page.page}")
        for line in page.lines:
            print(f"Line: '{line.text}' (Confidence: {line.appearance.style.confidence if line.appearance else 'N/A'})")
            for word in line.words:
                print(f"  └─ Word: '{word.text}' (Confidence: {word.confidence})")
else:
    print("❌ OCR failed.")
```

---

### 🧪 Sample Output
(This sample output is certainly hypothetical. May you be blessed with the confidence of ChatGPT. -KGK)
``` python
📄 Page 1
Line: 'City: BELLINGHAM' (Confidence: high)
  └─ Word: 'City:' (Confidence: 0.998)
  └─ Word: 'BELLINGHAM' (Confidence: 0.963)
```

---

🔁 Optional Enhancements
- Export to CSV:  
    Add a csv.writer or pandas.DataFrame export to log words and confidences.
- Draw bounding boxes:  
    Use word.bounding_box to draw boxes on the image with OpenCV or PIL.
- PDF & multipage:  
    This API also works with PDFs, just change the file input.

In [ ]:
# Import dependencies
import os
import time
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes

# Note: CognitiveServiceCredentials is deprecated.
# Recommend we use `AzureKeyCredential` from `azure.core.credentials` via `pip install azure-ai-vision`, instead. -KGK
# from msrest.authentication import CognitiveServiceCredentials

# Get the Azure key and endpoint from local environment vars
azure_key = "" # TODO: store azure key securely
azure_endpoint = "" # TODO: store azure endpoint securely

# TODO: Initialize the client, perform OCR

ImportError: cannot import name 'CognitiveServiceCredentials' from 'msrest.authentication' (c:\Users\7hesa\.conda\envs\stormwater_exp\Lib\site-packages\msrest\authentication.py)